<a href="https://colab.research.google.com/github/rajkumark30/Agentic_AI_Training/blob/main/Lab3_1_LangGraph_Persistence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3.1: LangGraph Persistence (Banking Assistant)

In this lab, we will add **Persistence** (Memory) to our graph. This allows the graph to remember the state (e.g., account context) across different interactions, enabling long-running conversations.

## Key Concepts
1. **Checkpointer**: A mechanism to save the state of the graph at every step.
2. **MemorySaver**: An in-memory checkpointer for development/testing.
3. **Thread ID**: A unique identifier to separate different user sessions.

In [25]:
# 1. Install Dependencies
%pip install -qU langchain-groq langchain-community langgraph

In [26]:
# 2. Setup API Keys
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# Optional for LangSmith Tracking
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "RAG Basics"

### 3.1 Define State and Imports
First, we define the structure of our graph's state and import necessary libraries.

In [27]:
# Imports and State Definition
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]

### 3.2 Initialize LLM and Persona
We set up the ChatGroq model and define the system prompt for the Banking Assistant.

In [28]:
# Initialize LLM
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
)

# Banking Assistant Persona
sys_msg = SystemMessage(content="""you need to share the details of yesterdays criket match
You can answer questions about score, teams played, who won, who was the player of the match
1. If the user asks What is the score? always check what game whether its a cricket or football
2. Who won the match?
3. Which teams played?

- India vs Australia
Score - India: 180/2 / Australia: 172/8
Player of the match : Virat Kohli
Winner : India
""")

### 3.3 Define Node Function
Here we define the chatbot node. We add a print statement to see when this node is actually executed.

In [29]:
def chatbot(state: State):
    print("--- Node: Chatbot Activated ---")
    return {"messages": [llm.invoke([sys_msg] + state["messages"])]}

### 3.4 Verification: Test the Chatbot Node
Before building the graph, let's test the `chatbot` function directly to ensure it responds correctly.

In [30]:
# Verification: Test the node directly
print("Running verification...")
dummy_state = {"messages": [("user", "Hello, what can you do?")]}
response = chatbot(dummy_state)
print("Response:", response['messages'][0].content)

Running verification...
--- Node: Chatbot Activated ---
Response: Hello! I can share details about yesterday's cricket match between **India and Australia**. Here's what I can tell you:  
- **Score**: India 180/2, Australia 172/8  
- **Winner**: India  
- **Player of the Match**: Virat Kohli  

Ask me anything about the game! 🏏


### 3.5 Build the Graph
Now we assemble the nodes and edges into a `StateGraph`.

In [31]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

## 4. Add Persistence
We use `MemorySaver` to store the state.

In [32]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# Compile with checkpointer
graph = graph_builder.compile(checkpointer=memory)

## 5. Run with Thread ID
We use `configurable` to specify the thread. In this initial interaction, we establish the context (Checking Account).

In [33]:
config = {"configurable": {"thread_id": "1"}}

# User establishes context
user_input = "What is the score? and who is the player of the match?"

for event in graph.stream({"messages": [("user", user_input)]}, config=config):
    for value in event.values():
        print("Assistant:", value["messages"][-1].content)

--- Node: Chatbot Activated ---
Assistant: **Score:**  
- **India:** 180/2  
- **Australia:** 172/8  

**Player of the Match:** **Virat Kohli** (India)  

**Winner:** India defeated Australia in the match.  
**Teams Played:** India vs Australia.


## 6. Verify Memory (Persistence)
Now we ask "What is the balance?" **without specifying the account**. The bot should remember we are talking about **Checking** because of the shared `thread_id`.

In [34]:
user_input = "What is the score?"

# Note: We are using the same 'config' with thread_id='1'
for event in graph.stream({"messages": [("user", user_input)]}, config=config):
    for value in event.values():
        print("Assistant:", value["messages"][-1].content)

--- Node: Chatbot Activated ---
Assistant: **Score:**  
- **India:** 180/2  
- **Australia:** 172/8  

**Winner:** India won the match.  
**Player of the Match:** Virat Kohli (India).


## 7. New Thread (No Memory)
If we change the `thread_id` to "2", the memory should be empty. The bot should NOT know which account we are referring to.

In [40]:
config_new = {"configurable": {"thread_id": "5"}}

# Asking the same question in a new thread
user_input = "you're not smart"

for event in graph.stream({"messages": [("user", user_input)]}, config=config_new):
    for value in event.values():
        print("Assistant:", value["messages"][-1].content)

--- Node: Chatbot Activated ---
Assistant: I'm here to help and provide accurate information based on what I know. If there's something specific you're looking for that I haven't addressed, let me know, and I'll do my best to assist! 😊 What would you like to know about the match?


In [41]:
user_input = "cricket  match."

# Note: We are using the same 'config' with thread_id='1'
for event in graph.stream({"messages": [("user", user_input)]}, config=config_new):
    for value in event.values():
        print("Assistant:", value["messages"][-1].content)

--- Node: Chatbot Activated ---
Assistant: Let me know what specific details you'd like about the **India vs Australia** cricket match! For example:  
- **Player stats** (e.g., Virat Kohli's performance)?  
- **Key moments** from the game?  
- **Tournament context** (e.g., series standings)?  
- **Upcoming matches**?  

I’m here to help—just ask! 🏏
